Helper — Buscar arquivos pela Landing

In [1]:
import pandas as pd

from src.config.settings import Settings
from src.bronze.catalog import BronzeCatalog

landing_metadata_df = pd.read_csv(Settings.LANDING_METADATA_PATH)


def get_file_by_name(source_file: str):
    """
    Busca o arquivo mais recente no landing_metadata.csv.

    Isso evita hardcode de caminho e snapshot no notebook.
    """
    filtered_df = landing_metadata_df[
        landing_metadata_df["source_file"] == source_file
    ]

    if filtered_df.empty:
        raise ValueError(f"Arquivo não encontrado: {source_file}")

    return filtered_df.sort_values("snapshot_date").iloc[-1].to_dict()


def get_latest_nact_by_extension(file_extension: str):
    """
    Busca o NACT mais recente pela extensão do arquivo.

    Exemplo:
    - .xlsb para EY
    - .xlsx para Deloitte
    """
    filtered_df = landing_metadata_df[
        (landing_metadata_df["relative_path"].str.contains("NACT", na=False))
        & (landing_metadata_df["file_extension"] == file_extension)
    ]

    if filtered_df.empty:
        raise ValueError(f"NACT não encontrado para extensão: {file_extension}")

    return filtered_df.sort_values("snapshot_date").iloc[-1].to_dict()

In [11]:
dir()

['BronzeCatalog',
 'BronzePipeline',
 'In',
 'Out',
 'Settings',
 '_',
 '_10',
 '_8',
 '_9',
 '__',
 '___',
 '__builtin__',
 '__builtins__',
 '__doc__',
 '__loader__',
 '__name__',
 '__package__',
 '__spec__',
 '_dh',
 '_i',
 '_i1',
 '_i10',
 '_i11',
 '_i2',
 '_i3',
 '_i4',
 '_i5',
 '_i6',
 '_i7',
 '_i8',
 '_i9',
 '_ih',
 '_ii',
 '_iii',
 '_oh',
 'bronze_batch_df',
 'exit',
 'get_file_by_name',
 'get_ipython',
 'get_latest_nact_by_extension',
 'landing_metadata_df',
 'nact_files_df',
 'open',
 'pd',
 'pipeline',
 'quit',
 'row',
 'run_bronze_batch',
 'spark',
 'xlsb_output_path']

1. Objective

2. Setup

In [2]:
from src.bronze.pipeline import BronzePipeline

pipeline = BronzePipeline()
spark = pipeline.spark

26/06/25 21:41:33 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/06/25 21:41:35 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


3. Validando CSV

In [12]:
row = get_file_by_name("ControleMedicoesPagamentos.csv")

csv_output_path = pipeline.run(
    source_path=row["source_path"],
    source_type="csv",
    dataset_name="controle_medicoes_pagamentos",
    snapshot_date=row["snapshot_date"],
    source_file=row["source_file"],
    file_hash=row["file_hash"],
)

print(csv_output_path)
spark.read.parquet(csv_output_path).select(
    "_snapshot_date",
    "_source_file",
    "_source_type",
).show(5)

s3a://contracts/bronze/source_type=csv/dataset=controle_medicoes_pagamentos/snapshot_date=2024-02-01_0800/
+---------------+--------------------+------------+
| _snapshot_date|        _source_file|_source_type|
+---------------+--------------------+------------+
|2024-02-01_0800|ControleMedicoesP...|         csv|
|2024-02-01_0800|ControleMedicoesP...|         csv|
|2024-02-01_0800|ControleMedicoesP...|         csv|
|2024-02-01_0800|ControleMedicoesP...|         csv|
|2024-02-01_0800|ControleMedicoesP...|         csv|
+---------------+--------------------+------------+
only showing top 5 rows



4. Validando XLSX

In [13]:
row = get_file_by_name("Exportação_bm_acompanhamento.xlsx")

xlsx_output_path = pipeline.run(
    source_path=row["source_path"],
    source_type="xlsx",
    dataset_name="controle_medicoes_andamento",
    snapshot_date=row["snapshot_date"],
    source_file=row["source_file"],
    file_hash=row["file_hash"],
)

print(xlsx_output_path)
spark.read.parquet(xlsx_output_path).select(
    "_snapshot_date",
    "_source_file",
    "_source_type",
).show(5)

s3a://contracts/bronze/source_type=xlsx/dataset=controle_medicoes_andamento/snapshot_date=2024-02-01_0800/
+---------------+--------------------+------------+
| _snapshot_date|        _source_file|_source_type|
+---------------+--------------------+------------+
|2024-02-01_0800|Exportação_bm_aco...|        xlsx|
|2024-02-01_0800|Exportação_bm_aco...|        xlsx|
|2024-02-01_0800|Exportação_bm_aco...|        xlsx|
|2024-02-01_0800|Exportação_bm_aco...|        xlsx|
|2024-02-01_0800|Exportação_bm_aco...|        xlsx|
+---------------+--------------------+------------+
only showing top 5 rows



5. Validando XLSB / NACT

In [6]:
nact_files_df = landing_metadata_df[
    (landing_metadata_df["relative_path"].str.contains("/NACT/", na=False))
    & (landing_metadata_df["file_extension"] == ".xlsb")
]

row = nact_files_df.sort_values(
    "snapshot_date"
).iloc[-1].to_dict()

xlsb_output_path = pipeline.run(
    source_path=row["source_path"],
    source_type="xlsb",
    dataset_name="nact",
    snapshot_date=row["snapshot_date"],
    source_file=row["source_file"],
    file_hash=row["file_hash"],
)

print(xlsb_output_path)
spark.read.parquet(xlsb_output_path).select(
    "_snapshot_date",
    "_source_file",
    "_source_type",
).show(5)

26/06/25 21:42:00 WARN MetricsConfig: Cannot locate configuration: tried hadoop-metrics2-s3a-file-system.properties,hadoop-metrics2.properties
26/06/25 21:42:01 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.
26/06/25 21:42:04 WARN TaskSetManager: Stage 0 contains a task of very large size (1047 KiB). The maximum recommended task size is 1000 KiB.


s3a://contracts/bronze/source_type=xlsb/dataset=nact/snapshot_date=2024-02-01_0800/
+---------------+-----------------+------------+
| _snapshot_date|     _source_file|_source_type|
+---------------+-----------------+------------+
|2024-02-01_0800|202402_ADMIN.xlsb|        xlsb|
|2024-02-01_0800|202402_ADMIN.xlsb|        xlsb|
|2024-02-01_0800|202402_ADMIN.xlsb|        xlsb|
|2024-02-01_0800|202402_ADMIN.xlsb|        xlsb|
|2024-02-01_0800|202402_ADMIN.xlsb|        xlsb|
+---------------+-----------------+------------+
only showing top 5 rows



6. Batch Validation

In [8]:
from src.bronze.batch import run_bronze_batch

bronze_batch_df = run_bronze_batch(force_reprocess=False)

bronze_batch_df["status"].value_counts()

status
SKIPPED    39
Name: count, dtype: int64

7. Validar o log persistido:

In [9]:
from src.config.settings import Settings
import pandas as pd

pd.read_csv(Settings.BRONZE_EXECUTION_LOG_PATH).head(10)

,execution_id,source_file,dataset_name,status,start_time,end_time,duration_seconds,error_message,snapshot_date,source_type,file_hash,bronze_path,retry_count
0,bronze_20260624_145226_7600ba06,ControleMedicoesPagamentos.csv,controle_medicoes_pagamentos,SUCCESS,2026-06-24T14:52:26.464756+00:00,2026-06-24T14:52:27.413885+00:00,0.949129,NaN,2024-07-13_0800,csv,b73d09f9dc6766eb918f3e1149649f758e0338d0f42f86...,s3a://contracts/bronze/source_type=csv/dataset...,NaN
1,bronze_20260624_145245_cf1a7458,ControleMedicoesPagamentos.csv,controle_medicoes_pagamentos,SKIPPED,2026-06-24T14:52:50.189023+00:00,2026-06-24T14:52:50.191407+00:00,0.002384,NaN,2024-07-13_0800,csv,b73d09f9dc6766eb918f3e1149649f758e0338d0f42f86...,NaN,NaN
2,bronze_20260624_145245_cf1a7458,Exportação_bm_acompanhamento.xlsx,controle_medicoes_andamento,SUCCESS,2026-06-24T14:52:50.191624+00:00,2026-06-24T14:53:01.947724+00:00,11.756100,NaN,2024-07-13_0800,xlsx,84df6ac3cb58f7ca789bf83e192ad99369bb1d35c6b41f...,s3a://contracts/bronze/source_type=xlsx/datase...,NaN
3,bronze_20260624_145245_cf1a7458,202211_ADMIN.xlsb,nact,SUCCESS,2026-06-24T14:53:01.947961+00:00,2026-06-24T14:53:14.508814+00:00,12.560853,NaN,2024-07-13_0800,xlsb,5d05acde002b287d9068b5b4b8dc34dc04609d1cc16cbb...,s3a://contracts/bronze/source_type=xlsb/datase...,NaN
4,bronze_20260624_145245_cf1a7458,Pendências-010223.xlsx,pendências_010223,SUCCESS,2026-06-24T14:53:14.509125+00:00,2026-06-24T14:53:15.560370+00:00,1.051245,NaN,2024-07-13_0800,xlsx,3cca4c48eaa038cee8bd806f6cae89ca1de1e6ee0e9996...,s3a://contracts/bronze/source_type=xlsx/datase...,NaN
5,bronze_20260624_145245_cf1a7458,QEC_5900055119_7_55_77.csv,qec,SUCCESS,2026-06-24T14:53:15.560937+00:00,2026-06-24T14:53:16.971908+00:00,1.410971,NaN,2024-07-13_0800,csv,f8a0265c96ffd7085b4e65eff75c24ba5d3439daeabbc2...,s3a://contracts/bronze/source_type=csv/dataset...,NaN
6,bronze_20260624_145245_cf1a7458,QEC_5900074722_7_55_77.csv,qec,SUCCESS,2026-06-24T14:53:16.972197+00:00,2026-06-24T14:53:18.192436+00:00,1.220239,NaN,2024-07-13_0800,csv,705844e05903b043b8ffcbe1f4889db5e08aea5370d378...,s3a://contracts/bronze/source_type=csv/dataset...,NaN
7,bronze_20260624_145245_cf1a7458,QEC_5900083950_7_55_77.csv,qec,SUCCESS,2026-06-24T14:53:18.192738+00:00,2026-06-24T14:53:19.136027+00:00,0.943289,NaN,2024-07-13_0800,csv,e9d67a12cd2f3754ab7e647f0e330ff373509e7fe74643...,s3a://contracts/bronze/source_type=csv/dataset...,NaN
8,bronze_20260624_145245_cf1a7458,QEC_5900086165_7_55_77.csv,qec,SUCCESS,2026-06-24T14:53:19.136501+00:00,2026-06-24T14:53:20.111475+00:00,0.974974,NaN,2024-07-13_0800,csv,acf1d09ec8d814ce474f28a371b3b2d885d5c183130b9c...,s3a://contracts/bronze/source_type=csv/dataset...,NaN
9,bronze_20260624_145245_cf1a7458,QEC_5900086169_7_55_77.csv,qec,SUCCESS,2026-06-24T14:53:20.111709+00:00,2026-06-24T14:53:21.011548+00:00,0.899839,NaN,2024-07-13_0800,csv,5978cdd5de1fa7b267ff214b6e0e4f56b3b310810fdf9b...,s3a://contracts/bronze/source_type=csv/dataset...,NaN


8. Quality Log / Reject Tracking

Objetivo: registrar problemas técnicos durante leitura
sem quebrar o batch inteiro

In [10]:
# Validação
bronze_batch_df = run_bronze_batch(force_reprocess=True)

bronze_batch_df["status"].value_counts()

26/06/25 21:42:59 WARN TaskSetManager: Stage 6 contains a task of very large size (1043 KiB). The maximum recommended task size is 1000 KiB.
Skipping line 479: Expected 13 fields in line 479, saw 15                       
26/06/25 21:43:26 WARN TaskSetManager: Stage 19 contains a task of very large size (1047 KiB). The maximum recommended task size is 1000 KiB.
Skipping line 479: Expected 13 fields in line 479, saw 15                       
26/06/25 21:43:49 WARN TaskSetManager: Stage 32 contains a task of very large size (1047 KiB). The maximum recommended task size is 1000 KiB.
Skipping line 479: Expected 13 fields in line 479, saw 15


status
SUCCESS    39
Name: count, dtype: int64

In [ ]:
from src.config.settings import Settings
import pandas as pd

quality_log_df = pd.read_csv(Settings.BRONZE_QUALITY_LOG_PATH)

quality_log_df.tail(10)

9. Processed Files Manifest validation

In [ ]:
from src.bronze.batch import run_bronze_batch

bronze_batch_df = run_bronze_batch(force_reprocess=False)

bronze_batch_df["status"].value_counts()

In [ ]:
bronze_batch_df = run_bronze_batch(force_reprocess=True)
bronze_batch_df["status"].value_counts()

In [ ]:
from src.bronze.batch import run_bronze_batch

bronze_batch_df = run_bronze_batch(force_reprocess=False)
bronze_batch_df["status"].value_counts()

10. Sumário

In [ ]:
bronze_validation = {
    "csv_path": csv_output_path,
    "xlsx_path": xlsx_output_path,
    "xlsb_path": xlsb_output_path,
    "batch_status": bronze_batch_df["status"].value_counts().to_dict(),
    "status": "SUCCESS",
}

bronze_validation

11. Stop Spark - Isso libera o worker para o próximo notebook.

In [ ]:
spark.stop()